
## **PHASE 4: FULL-DATASET INFERENCE**

The goal of this phase is to apply trained DistilBERT model to the whoel dataset.
Run on GPU (Google Colab)

**Model Inference:**
Using the saved model checkpoint (aspect_sentiment_model_final.pt), the pipeline processes unannotated reviews to automatically predict sentiment (not mentioned, positive, or negative) across all seven aspects at scale.


### Step 1: Load model and tokenizer

In [1]:
import torch
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizerFast, DistilBertModel
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report
import random
import numpy as np


### Step 2: Define multihead classifier

In [ ]:
# Define multi-head classifier using DistilBERT plus one linear head per aspect, same like the training loop.
class MultiAspectClassifier(nn.Module):
    def __init__(self, model_name, aspects, num_classes=3, dropout=0.3):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained(model_name)
        self.aspects = aspects
        hidden_size = self.bert.config.hidden_size

        self.dropout = nn.Dropout(dropout)
        self.heads = nn.ModuleDict({
            aspect: nn.Linear(hidden_size, num_classes) for aspect in aspects
        })

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        pooled = self.dropout(pooled)

        logits = {aspect: self.heads[aspect](pooled) for aspect in self.aspects}
        return logits

In [ ]:
# Set the device to GPU if available, otherwise use CPU or MPS (for Apple Silicon)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Using device for inference:", DEVICE)
MODEL_NAME = "distilbert-base-uncased"
ASPECTS = ["texture", "scent", "breakouts", "irritation", "efficacy", "packaging", "value"]
MAX_LEN = 256

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

model = MultiAspectClassifier(MODEL_NAME, ASPECTS).to(DEVICE)
checkpoint = torch.load("aspect_sentiment_model_final.pt", map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print("Model loaded.")

Using device for inference: cuda


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded.


### Step 3: Load the full dataset

In [4]:
import pandas as pd

df_full = pd.read_csv("skincare_reviews_clean.csv")
print(df_full.shape)

(940641, 15)


In [ ]:
# Define a custom Dataset class for inference.
class InferenceDataset(Dataset):
    def __init__(self, texts):
        self.texts = texts

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        # Return raw text instead of pre-padded tensors
        return str(self.texts[idx])

# Collate function handles dynamic padding per batch to reduce memory usage and improve efficiency during inference.
def collate_fn(batch):
    return tokenizer(
        batch,
        truncation=True,
        padding=True,       # Automatically pads only to the max length in this batch
        max_length=MAX_LEN,
        return_tensors="pt"
    )

# Run inference on the entire dataset and return predictions for each aspect.
def run_inference(df, text_col="review_text", batch_size=128):
    dataset = InferenceDataset(df[text_col].tolist())

    # Pass collate_fn to the DataLoader so it batches and pads dynamically
    loader = DataLoader(dataset, batch_size=batch_size, collate_fn=collate_fn)

    all_preds = {aspect: [] for aspect in ASPECTS}

    model.eval()
    with torch.no_grad():
        for batch in tqdm(loader):
            # batch is the dictionary directly returned by the tokenizer collate function
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            logits = model(input_ids, attention_mask)

            for aspect in ASPECTS:
                preds = torch.argmax(logits[aspect], dim=1).cpu().numpy()
                all_preds[aspect].extend(preds)

    return all_preds

### Step 4: Run on the small sample

In [ ]:
# Run inference on a sample of 500 reviews to check predictions
sample = df_full.sample(500, random_state=1).reset_index(drop=True)
preds_sample = run_inference(sample, batch_size=64)

for aspect in ASPECTS:
    sample[f"{aspect}_pred"] = preds_sample[aspect]

sample[["review_text"] + [f"{a}_pred" for a in ASPECTS]].head(10)

100%|██████████| 8/8 [00:03<00:00,  2.01it/s]


,review_text,texture_pred,scent_pred,breakouts_pred,irritation_pred,efficacy_pred,packaging_pred,value_pred
0,This was gifted by Ren skincare in exchange fo...,1,0,0,0,1,0,0
1,I got this as my birthday gift & LOVED IT! Cle...,1,1,0,0,1,0,0
2,I’ve been using this product for about a week ...,1,0,0,0,1,1,0
3,I’m always looking for a new eye serum that wi...,1,0,0,0,1,0,0
4,I got the watermelon one but it won’t let me c...,0,1,0,0,0,0,2
5,You have to try this! You won’t be sorry!!,0,0,0,0,0,0,0
6,I received this oh about 2 weeks ago now and u...,0,0,0,0,1,0,0
7,At first I loved this.. so hydrating and I hav...,2,0,0,0,2,0,0
8,This stuff is pretty great. It comes in a tiny...,0,0,0,0,1,0,0
9,This is a super lightweight moisturizer that’s...,1,0,0,0,1,0,1


### Step 5: Run inference on the full dataset

Now that the smoke test works, we run inference on every review in the cleaned dataset.

The model predicts one sentiment label for each of the seven aspects:

- Texture
- Scent
- Breakouts
- Irritation
- Efficacy
- Packaging
- Value

Predictions are processed in batches to avoid running out of memory.

In [ ]:
# Run inference on the entire dataset
preds_full = run_inference(df_full, batch_size=128)

print("Inference completed.")
print(f"Processed {len(df_full):,} reviews.")

100%|██████████| 7349/7349 [1:55:12<00:00,  1.06it/s]

Inference completed.
Processed 940,641 reviews.


### Step 6: Convert numeric predictions back to sentiment labels

The model outputs class IDs:

- `0` → Not mentioned
- `1` → Positive
- `2` → Negative

These IDs are converted back into human-readable labels before saving the results.

In [ ]:
# Convert class IDs back to sentiment labels
id2label = {
    0: "Not mentioned",
    1: "Positive",
    2: "Negative"
}

for aspect in ASPECTS:
    df_full[f"{aspect}_pred"] = [
        id2label[p] for p in preds_full[aspect]
    ]

print("Prediction labels added.")

Prediction labels added.


### Step 7: Inspect the prediction results

Before saving the dataset, we perform a few quick checks to make sure the predictions look reasonable.

For each aspect, we inspect:

- the distribution of predicted classes
- a few example reviews with their predicted sentiments

In [ ]:
# Check prediction distribution for each aspect
for aspect in ASPECTS:
    print(f"\n{'='*20} {aspect.upper()} {'='*20}")
    print(df_full[f"{aspect}_pred"].value_counts())


==================== TEXTURE ====================
texture_pred
Positive         540003
Not mentioned    325552
Negative          75086
Name: count, dtype: int64

==================== SCENT ====================
scent_pred
Not mentioned    700496
Positive         209132
Negative          31013
Name: count, dtype: int64

==================== BREAKOUTS ====================
breakouts_pred
Not mentioned    871714
Positive          44806
Negative          24121
Name: count, dtype: int64

==================== IRRITATION ====================
irritation_pred
Not mentioned    848303
Positive          61885
Negative          30453
Name: count, dtype: int64

==================== EFFICACY ====================
efficacy_pred
Positive         754264
Not mentioned    124622
Negative          61755
Name: count, dtype: int64

==================== PACKAGING ====================
packaging_pred
Not mentioned    861458
Positive          43034
Negative          36149
Name: count, dtype: int64

===============

In [ ]:
# Show a few example predictions
example_columns = [
    "review_text",
    "texture_pred",
    "scent_pred",
    "breakouts_pred",
    "irritation_pred",
    "efficacy_pred",
    "packaging_pred",
    "value_pred"
]

df_full[example_columns].sample(5, random_state=42)

,review_text,texture_pred,scent_pred,breakouts_pred,irritation_pred,efficacy_pred,packaging_pred,value_pred
14652,"At first, the product was very good and I foun...",Not mentioned,Not mentioned,Not mentioned,Not mentioned,Negative,Not mentioned,Negative
693413,I received a small sample size for testing pur...,Not mentioned,Not mentioned,Not mentioned,Not mentioned,Positive,Not mentioned,Not mentioned
770665,Really want to like this product but not thril...,Negative,Not mentioned,Not mentioned,Not mentioned,Negative,Not mentioned,Not mentioned
535826,I don’t love strong scents in my nighttime ski...,Positive,Negative,Not mentioned,Not mentioned,Positive,Not mentioned,Not mentioned
610232,Purchased today and wondering if the smell goe...,Not mentioned,Negative,Not mentioned,Not mentioned,Not mentioned,Not mentioned,Not mentioned


### Step 8: Save the prediction dataset

The seven predicted sentiment columns are added to the original dataframe and saved as a new CSV file.

This file will be used in the next phase for business analysis and visualization.

In [11]:
# Save the dataframe with predictions

output_path = "skincare_reviews_with_predictions.csv"

df_full.to_csv(output_path, index=False)

print(f"Saved predictions to: {output_path}")
print(f"Rows: {len(df_full):,}")

Saved predictions to: skincare_reviews_with_predictions.csv
Rows: 940,641


### Step 9: Verify the saved dataset

Reload the saved file and confirm that:

- the file was written correctly,
- the prediction columns exist,
- the dataset size matches the original.

In [ ]:
# Reload the saved file for verification

df_check = pd.read_csv(output_path)

print(df_check.shape)

prediction_columns = [f"{aspect}_pred" for aspect in ASPECTS]
print(df_check[prediction_columns].head(5))

/tmp/ipykernel_861/3966629780.py:3: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_check = pd.read_csv(output_path)


(940641, 22)
    texture_pred     scent_pred breakouts_pred irritation_pred  efficacy_pred  \
0       Positive  Not mentioned  Not mentioned   Not mentioned       Positive   
1  Not mentioned  Not mentioned  Not mentioned   Not mentioned       Negative   
2  Not mentioned  Not mentioned  Not mentioned   Not mentioned       Positive   
3       Positive       Positive  Not mentioned   Not mentioned  Not mentioned   
4  Not mentioned  Not mentioned  Not mentioned   Not mentioned  Not mentioned   

  packaging_pred     value_pred  
0  Not mentioned  Not mentioned  
1  Not mentioned  Not mentioned  
2  Not mentioned  Not mentioned  
3  Not mentioned  Not mentioned  
4  Not mentioned       Negative  


### Step 10: Aggregate predictions to product x skin_type level 

In [ ]:
import pandas as pd

df_full = pd.read_csv("skincare_reviews_with_predictions.csv", dtype={"author_id": str}, low_memory=False)

ASPECTS = ["texture", "scent", "breakouts", "irritation", "efficacy", "packaging", "value"]

# Aggregate predictions to product x skin_type level
def aggregate_aspect(df, aspect, group_cols=["product_id", "product_name", "skin_type"]):
    col = f"{aspect}_pred"
    counts = df.groupby(group_cols)[col].value_counts().unstack(fill_value=0)

    # Ensure all three categories exist even if a group has zero of one
    for cat in ["Positive", "Negative", "Not mentioned"]:
        if cat not in counts.columns:
            counts[cat] = 0

    counts["total_mentions"] = counts["Positive"] + counts["Negative"]
    counts["pct_positive"] = (counts["Positive"] / counts["total_mentions"]).where(counts["total_mentions"] > 0)
    counts["pct_negative"] = (counts["Negative"] / counts["total_mentions"]).where(counts["total_mentions"] > 0)

    counts = counts.rename(columns={
        "Positive": f"{aspect}_positive_count",
        "Negative": f"{aspect}_negative_count",
        "Not mentioned": f"{aspect}_not_mentioned_count",
        "total_mentions": f"{aspect}_total_mentions",
        "pct_positive": f"{aspect}_pct_positive",
        "pct_negative": f"{aspect}_pct_negative",
    })
    return counts.reset_index()

# Build the summary table by merging each aspect's aggregation together
summary = None
for aspect in ASPECTS:
    agg = aggregate_aspect(df_full, aspect)
    if summary is None:
        summary = agg
    else:
        summary = summary.merge(agg, on=["product_id", "product_name", "skin_type"], how="outer")

print(summary.shape)
summary.head()

# The summary table is now ready for analysis. 
# It contains aggregated counts and percentages of positive, negative, and not mentioned sentiments for each aspect, grouped by product and skin type.

(9000, 45)


,product_id,product_name,skin_type,texture_negative_count,texture_not_mentioned_count,texture_positive_count,texture_total_mentions,texture_pct_positive,texture_pct_negative,scent_negative_count,...,packaging_positive_count,packaging_total_mentions,packaging_pct_positive,packaging_pct_negative,value_negative_count,value_not_mentioned_count,value_positive_count,value_total_mentions,value_pct_positive,value_pct_negative
0,P107306,Renewing Eye Cream,Combination,8,65,37,45,0.822222,0.177778,1,...,1,6,0.166667,0.833333,25,73,12,37,0.324324,0.675676
1,P107306,Renewing Eye Cream,Dry,3,23,14,17,0.823529,0.176471,0,...,0,2,0.000000,1.000000,7,31,2,9,0.222222,0.777778
2,P107306,Renewing Eye Cream,Normal,0,18,9,9,1.000000,0.000000,0,...,0,3,0.000000,1.000000,1,25,1,2,0.500000,0.500000
3,P107306,Renewing Eye Cream,Oily,4,12,6,10,0.600000,0.400000,0,...,0,2,0.000000,1.000000,8,12,2,10,0.200000,0.800000
4,P107306,Renewing Eye Cream,Unknown,2,35,15,17,0.882353,0.117647,1,...,1,1,1.000000,0.000000,8,41,3,11,0.272727,0.727273


In [ ]:
# Save the summary table 
summary.to_csv("aspect_sentiment_summary.csv", index=False)
print("Saved aspect_sentiment_summary.csv:", summary.shape)

Saved aspect_sentiment_summary.csv: (9000, 45)


## Summary of Phase 4: Full-Dataset Inference

### What I did in this phase
In this notebook, I ran inference across my entire cleaned skincare review dataset consisting of roughly 940,000 reviews. I loaded my trained multi-aspect sentiment model, passed all the reviews through it to predict sentiment labels (positive, negative, or not mentioned) for seven different aspects (texture, scent, breakouts, irritation, efficacy, packaging, and value), and then aggregated these predictions down to the product and skin-type level for downstream use.

### Methodology & Choices
* **Inference Pipeline & Dynamic Padding:** Instead of padding every review to a fixed maximum length, I used a custom `collate_fn` to pad batches dynamically based on the length of the longest review in each specific batch. This significantly optimized memory usage and processing speed.
* **Why Batch Size 128?** I chose a batch size of 128 because it strikes the ideal balance—it maximizes GPU throughput and speeds up execution across nearly a million rows without causing a CUDA Out-of-Memory (OOM) error.
* **Running on Google Colab (GPU):** I executed this notebook on Google Colab using a T4 GPU. Running inference on ~940k transformer predictions on a local CPU would have taken many hours or even days, whereas the GPU crunched through the entire dataset in a reasonable amount of time.

### Evaluating the Results
* **Prediction Distributions:** The model predictions show realistic real-world distributions where the vast majority of aspects are labeled as "Not mentioned," which aligns with how people actually write reviews (they usually only comment on specific features if they feel strongly about them).
* **Aggregated Summary:** Checking the final summary table confirms that the aggregated sentiment counts and percentages per skin type look sensible and are ready to be used for app features.

## Limitations, Improvements, and Future Work

### Limitations of the Current Approach
* **Propagation of Training Bias:** Because my training dataset in earlier phases was intentionally augmented with targeted negative samples (especially for *packaging* and *irritation* to fix class scarcity), the model can be slightly biased toward predicting negative packaging on the full dataset. Therefore, packaging negative percentages should be treated as approximate estimates rather than absolute ground truths.
* **Token Truncation Length:** I set the maximum token length to 256 for inference. While this covers the vast majority of reviews, any sentiment or nuance expressed very late in exceptionally long reviews gets cut off and missed by the model.
* **Mention-Based Metrics:** The aspect sentiment percentages (e.g., `% positive` or `% negative`) are calculated strictly among reviews where that specific aspect was *predicted as mentioned*. They do not represent all reviews for a product as a whole, meaning a product might have high positive percentages for texture, but only because very few people actually bothered to comment on its texture.
* **Inference Noise:** These are model-generated pre-computed estimates, not verified human labels. While the model performed well on the gold set, some individual predictions on edge-case reviews might still contain errors.

### Potential Improvements
* **Architecture Upgrades:** I could experiment with a stronger encoder backbone (like `DeBERTa-v3` or domain-specific models) to see if it helps boost performance on tricky, nuanced aspects like *texture* and *efficacy*, which had lower F1 scores due to descriptive vs. sentimental overlap.
* **Longer Context Windows:** Increasing `MAX_LEN` from 256 to 512 tokens could capture information from longer, more detailed reviews without truncation, though it would require more GPU memory and compute time.
* **Confidence Thresholding:** Instead of blindly accepting the `argmax` prediction, I could implement a probability threshold (e.g., only counting an aspect as "mentioned" if the model's confidence exceeds a certain level) to reduce false positives for neutral mentions.

### Future Work
* **Real-Time / Incremental Pipeline:** Expanding the pipeline so that newly scraped reviews can be automatically ingested, run through the model, and update product summaries dynamically.
* **Aspect-Based Recommendation System:** Moving beyond simple sentiment aggregation to build a recommendation engine that suggests skincare products based on a user's specific skin concerns (e.g., recommending products specifically optimized for low breakouts and high value).